# Embedding Layer Analysis

Analyze the embedding layer: norm structure, embed-unembed alignment,
neighborhoods, positional embeddings, and effective dimensionality.

## Why This Matters

Layer embedding characterizes what each layer contributes to the model's computation. Understanding layer-level organization — which layers are critical, which are redundant, and how they specialize — is essential for both interpretability and efficient model design.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.embedding_layer_analysis import (
    embedding_norm_structure, embedding_similarity_to_unembed,
    embedding_neighborhood, positional_embedding_structure,
    embedding_effective_dimension,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Norm Structure

Are input token embeddings similar in norm, or do some stand out?

In [ ]:
result = embedding_norm_structure(model, tokens)
print(f"Global: mean={result['global_mean_norm']:.4f} ± {result['global_std_norm']:.4f}")
print(f"Range: [{result['global_min_norm']:.4f}, {result['global_max_norm']:.4f}]\n")
for p in result['per_token']:
    outlier = ' [OUTLIER]' if p['is_outlier'] else ''
    bar = '█' * int(p['percentile'] * 20)
    print(f"  Pos {p['position']} (tok {p['token']:2d}): norm={p['norm']:.4f}, "
          f"pctl={p['percentile']:.1%}{outlier} {bar}")

## Embed-Unembed Alignment

Do input embeddings align with their corresponding output directions?

In [ ]:
result = embedding_similarity_to_unembed(model, tokens)
print(f"Mean cos(embed, unembed): {result['mean_embed_unembed_cos']:.4f}\n")
for p in result['per_token']:
    promote = '+' if p['is_self_promoting'] else '-'
    print(f"  Pos {p['position']} (tok {p['token']:2d}): "
          f"cos={p['embed_unembed_cos']:+.4f}, "
          f"self_logit={p['self_logit']:+.4f} [{promote}]")

## Embedding Neighborhoods

Which tokens are nearest neighbors in embedding space?

In [ ]:
result = embedding_neighborhood(model, tokens, top_k=5)
for p in result['per_token']:
    neighbors = ', '.join(f"tok{n['token']}({n['similarity']:.3f})" for n in p['neighbors'])
    print(f"  Pos {p['position']} (tok {p['token']:2d}): mean_sim={p['mean_neighbor_sim']:.4f}")
    print(f"    neighbors: {neighbors}")

## Positional Embedding Structure

Norm, similarity, and content-position interaction.

In [ ]:
result = positional_embedding_structure(model, tokens)
print(f"Mean position similarity: {result['mean_position_similarity']:.4f}\n")
for p in result['per_position']:
    print(f"  Pos {p['position']}: norm={p['norm']:.4f}, "
          f"content_pos_cos={p['content_position_cos']:+.4f}")

## Effective Dimension

How much of the d_model space do the input embeddings span?

In [ ]:
result = embedding_effective_dimension(model, tokens)
print(f"Effective dim: {result['effective_dimension']:.2f} / {result['d_model']}")
print(f"Dims for 90%: {result['dim_for_90_pct']}\n")
for c in result['per_component']:
    bar = '█' * int(c['variance_explained'] * 40)
    print(f"  PC{c['component']}: sv={c['singular_value']:.4f}, "
          f"var_expl={c['variance_explained']:.1%}, "
          f"cumul={c['cumulative']:.1%} {bar}")